# 📓 GOOGLE COLAB NOTEBOOK

## JSON → Corporate PPT Generator

## 🟦 CELL 1 — Install Dependencies

In [54]:
!pip install python-pptx pydantic

## 🟦 CELL 2 — Imports & Constants

In [55]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE, XL_LEGEND_POSITION
import json
import os
from typing import List, Optional, Union, Literal, Dict, Any
from pydantic import BaseModel, Field, ValidationError, validator
from IPython.display import display, HTML

## 🟦 CELL 3 — Theme Definition (LOCKED)

> Single theme used for **all slides**

In [56]:
# ==========================================
# 🟦 DATA MODELS (STRICT VALIDATION)
# ==========================================
# These models ensure your JSON is correct before generating the PPT.
# They also define the structure for the "Theme" and "Slides".

class ThemeColors(BaseModel):
    background: List[int] = [255, 255, 255]
    title: List[int] = [47, 93, 140]
    text: List[int] = [31, 41, 51]
    accent: List[int] = [74, 144, 226]
    border: List[int] = [217, 221, 225]
    table_header: List[int] = [47, 93, 140]
    
class ThemeFonts(BaseModel):
    family: str = "Calibri"
    size_title: int = 36
    size_body: int = 18

class PresentationTheme(BaseModel):
    colors: ThemeColors = ThemeColors()
    fonts: ThemeFonts = ThemeFonts()

# --- Slide Content Models ---

class SlideCover(BaseModel):
    type: Literal["cover"]
    title: str
    subtitle: Optional[str] = ""

class SlideContent(BaseModel):
    type: Literal["content"]
    title: str
    banner: Optional[str] = None
    text: str

class TableColumn(BaseModel):
    key: str
    label: str

class SlideTable(BaseModel):
    type: Literal["table"]
    title: str
    columns: List[TableColumn]
    rows: List[Dict[str, Any]]

class ChartData(BaseModel):
    categories: List[str]
    series_name: str
    values: List[float]

class SlideChart(BaseModel):
    type: Literal["chart"]
    title: str
    chart: ChartData
    chart_type: Literal["COLUMN", "BAR", "LINE"] = "COLUMN"

class CardItem(BaseModel):
    title: str
    items: List[str]

class SlideCards(BaseModel):
    type: Literal["cards"]
    title: str
    cards: List[CardItem]

class GanttTask(BaseModel):
    name: str
    start_week: int
    duration: int
    progress: int = 0

class SlideGantt(BaseModel):
    type: Literal["gantt"]
    title: str
    tasks: List[GanttTask]

class SlideImage(BaseModel):
    type: Literal["image"]
    title: str
    image_path: str
    caption: Optional[str] = ""

# Union of all possible slide types
SlideType = Union[
    SlideCover, SlideContent, SlideTable, SlideChart, 
    SlideCards, SlideGantt, SlideImage
]

class PresentationMeta(BaseModel):
    title: str
    author: str
    date: str

class PresentationConfig(BaseModel):
    meta: PresentationMeta
    theme: PresentationTheme = PresentationTheme()
    slides: List[SlideType]

print("✅ Data Models Defined. Your JSON will now be strictly checked!")

✅ Data Models Defined. Your JSON will now be strictly checked!


## 🟦 CELL 4 — PPT Helper Functions

In [57]:
# ==========================================
# 🟦 RENDERER HELPERS
# ==========================================

def get_rgb(color_list):
    return RGBColor(color_list[0], color_list[1], color_list[2])

def add_common_elements(slide, title, theme: PresentationTheme):
    """Adds the title and footer to every slide."""
    # Title
    title_shape = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(12.33), Inches(1))
    tf = title_shape.text_frame
    p = tf.paragraphs[0]
    p.text = title
    p.font.size = Pt(theme.fonts.size_title)
    p.font.bold = True
    p.font.color.rgb = get_rgb(theme.colors.title)
    p.font.name = theme.fonts.family
    
    # Decorative Line
    line = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(1.1), Inches(12.33), Inches(0.05)
    )
    line.fill.solid()
    line.fill.fore_color.rgb = get_rgb(theme.colors.accent)
    line.line.fill.background()

def render_cover(slide, data: SlideCover, theme: PresentationTheme):
    # Background Color Block
    bg = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(0), Inches(13.33), Inches(7.5))
    bg.fill.solid()
    bg.fill.fore_color.rgb = get_rgb(theme.colors.title)
    
    # White Card Overlay
    card = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(2), Inches(2), Inches(9.33), Inches(3.5))
    card.fill.solid()
    card.fill.fore_color.rgb = get_rgb(theme.colors.background)
    
    # Title
    tb = slide.shapes.add_textbox(Inches(2.5), Inches(2.5), Inches(8.33), Inches(1.5))
    p = tb.text_frame.paragraphs[0]
    p.text = data.title
    p.font.size = Pt(54)
    p.font.bold = True
    p.font.color.rgb = get_rgb(theme.colors.title)
    p.alignment = PP_ALIGN.CENTER
    
    # Subtitle
    if data.subtitle:
        sb = slide.shapes.add_textbox(Inches(2.5), Inches(4.0), Inches(8.33), Inches(1))
        p = sb.text_frame.paragraphs[0]
        p.text = data.subtitle
        p.font.size = Pt(28)
        p.font.color.rgb = get_rgb(theme.colors.text)
        p.alignment = PP_ALIGN.CENTER

def render_content(slide, data: SlideContent, theme: PresentationTheme):
    top_cursor = 1.5
    
    if data.banner:
        box = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(0.5), Inches(top_cursor), Inches(12.33), Inches(1.2))
        box.fill.solid()
        box.fill.fore_color.rgb = get_rgb(theme.colors.accent)
        # Lighten the accent for background? Hard to do without HSL, so we use white text on accent
        
        tf = box.text_frame
        p = tf.paragraphs[0]
        p.text = data.banner
        p.font.size = Pt(20)
        p.font.color.rgb = RGBColor(255, 255, 255)
        p.alignment = PP_ALIGN.CENTER
        top_cursor += 1.5

    # Main Text
    tb = slide.shapes.add_textbox(Inches(0.5), Inches(top_cursor), Inches(12.33), Inches(5))
    tf = tb.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.text = data.text
    p.font.size = Pt(theme.fonts.size_body)
    p.font.color.rgb = get_rgb(theme.colors.text)
    p.font.name = theme.fonts.family

def render_table(slide, data: SlideTable, theme: PresentationTheme):
    rows = len(data.rows) + 1
    cols = len(data.columns)
    
    # Dynamic Width Calculation
    table_width = Inches(12.33)
    col_width = table_width / cols
    
    shape = slide.shapes.add_table(rows, cols, Inches(0.5), Inches(2.0), table_width, Inches(0.6 * rows))
    table = shape.table
    
    # Headers
    for i, col in enumerate(data.columns):
        cell = table.cell(0, i)
        cell.text = col.label
        cell.fill.solid()
        cell.fill.fore_color.rgb = get_rgb(theme.colors.table_header)
        p = cell.text_frame.paragraphs[0]
        p.font.bold = True
        p.font.color.rgb = RGBColor(255, 255, 255)
        p.alignment = PP_ALIGN.CENTER
        
    # Rows
    for r, row_data in enumerate(data.rows, start=1):
        for c, col in enumerate(data.columns):
            cell = table.cell(r, c)
            val = str(row_data.get(col.key, ""))
            cell.text = val
            p = cell.text_frame.paragraphs[0]
            p.font.size = Pt(14)
            p.font.color.rgb = get_rgb(theme.colors.text)
            p.alignment = PP_ALIGN.CENTER
            
            # Simple Status Coloring (Hardcoded logic for now, but could be dynamic)
            if val.lower() in ["completed", "approved", "on track"]:
                p.font.color.rgb = RGBColor(0, 150, 0)
                p.font.bold = True
            elif val.lower() in ["pending", "at risk", "delayed"]:
                p.font.color.rgb = RGBColor(200, 0, 0)
                p.font.bold = True

def render_chart(slide, data: SlideChart, theme: PresentationTheme):
    chart_data = CategoryChartData()
    chart_data.categories = data.chart.categories
    chart_data.add_series(data.chart.series_name, data.chart.values)
    
    chart_type = XL_CHART_TYPE.COLUMN_CLUSTERED
    if data.chart_type == "LINE": chart_type = XL_CHART_TYPE.LINE
    if data.chart_type == "BAR": chart_type = XL_CHART_TYPE.BAR_CLUSTERED
    
    chart = slide.shapes.add_chart(
        chart_type, Inches(0.5), Inches(2.0), Inches(12.33), Inches(4.5), chart_data
    ).chart
    
    chart.has_legend = True
    chart.legend.position = XL_LEGEND_POSITION.BOTTOM

def render_cards(slide, data: SlideCards, theme: PresentationTheme):
    count = len(data.cards)
    if count == 0: return
    
    margin = 0.5
    spacing = 0.3
    available_w = 13.33 - (2 * margin) - ((count - 1) * spacing)
    card_w = available_w / count
    
    for i, card in enumerate(data.cards):
        left = margin + (i * (card_w + spacing))
        top = 2.0
        
        # Card Box
        box = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(left), Inches(top), Inches(card_w), Inches(4.5))
        box.fill.solid()
        box.fill.fore_color.rgb = RGBColor(255, 255, 255)
        box.line.color.rgb = get_rgb(theme.colors.border)
        
        # Header Strip
        header = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(left), Inches(top), Inches(card_w), Inches(0.8))
        header.fill.solid()
        header.fill.fore_color.rgb = get_rgb(theme.colors.accent)
        
        # Title
        tf = header.text_frame
        p = tf.paragraphs[0]
        p.text = card.title
        p.font.bold = True
        p.font.color.rgb = RGBColor(255, 255, 255)
        p.alignment = PP_ALIGN.CENTER
        
        # Items
        content_box = slide.shapes.add_textbox(Inches(left + 0.1), Inches(top + 0.9), Inches(card_w - 0.2), Inches(3.5))
        for item in card.items:
            p = content_box.text_frame.add_paragraph()
            p.text = f"• {item}"
            p.font.size = Pt(14)
            p.font.color.rgb = get_rgb(theme.colors.text)
            p.space_after = Pt(10)

def render_gantt(slide, data: SlideGantt, theme: PresentationTheme):
    tasks = data.tasks
    if not tasks: return
    
    start_x = 3.0
    start_y = 2.5
    row_h = 0.6
    chart_w = 9.5
    
    min_w = min(t.start_week for t in tasks)
    max_w = max(t.start_week + t.duration for t in tasks)
    total_w = max_w - min_w + 1
    week_w = chart_w / total_w
    
    # Header
    for w in range(total_w):
        x = start_x + (w * week_w)
        lbl = slide.shapes.add_textbox(Inches(x), Inches(start_y - 0.4), Inches(week_w), Inches(0.4))
        p = lbl.text_frame.paragraphs[0]
        p.text = f"W{min_w + w}"
        p.font.size = Pt(10)
        p.alignment = PP_ALIGN.CENTER
        
        # Gridline
        line = slide.shapes.add_shape(MSO_SHAPE.LINE_INVERSE, Inches(x), Inches(start_y), Inches(0), Inches(len(tasks)*row_h))
        line.line.color.rgb = get_rgb(theme.colors.border)
        line.line.dash_style = 1

    # Tasks
    for i, task in enumerate(tasks):
        y = start_y + (i * row_h)
        
        # Label
        tb = slide.shapes.add_textbox(Inches(0.5), Inches(y), Inches(2.4), Inches(row_h))
        p = tb.text_frame.paragraphs[0]
        p.text = task.name
        p.font.size = Pt(12)
        p.alignment = PP_ALIGN.RIGHT
        
        # Bar
        bar_x = start_x + ((task.start_week - min_w) * week_w)
        bar_w = task.duration * week_w
        bar = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(bar_x), Inches(y+0.1), Inches(bar_w), Inches(row_h-0.2))
        bar.fill.solid()
        bar.fill.fore_color.rgb = get_rgb(theme.colors.accent)
        
        if task.progress > 0:
            p = bar.text_frame.paragraphs[0]
            p.text = f"{task.progress}%"
            p.font.size = Pt(9)
            p.font.color.rgb = RGBColor(255, 255, 255)
            p.alignment = PP_ALIGN.CENTER

def render_image(slide, data: SlideImage, theme: PresentationTheme):
    if os.path.exists(data.image_path):
        pic = slide.shapes.add_picture(data.image_path, Inches(1), Inches(2.0), height=Inches(4.5))
        pic.left = int((Inches(13.33) - pic.width) / 2)
    else:
        tb = slide.shapes.add_textbox(Inches(1), Inches(3), Inches(11), Inches(2))
        tb.text_frame.text = f"Image not found: {data.image_path}"
        
    if data.caption:
        tb = slide.shapes.add_textbox(Inches(1), Inches(6.6), Inches(11.33), Inches(0.5))
        p = tb.text_frame.paragraphs[0]
        p.text = data.caption
        p.font.italic = True
        p.alignment = PP_ALIGN.CENTER


## 🟦 CELL 5 — Table Renderer

## 🟦 CELL 6 — Card Renderer

## 🟦 CELL 7 — Timeline Renderer

## 🟦 CELL 8 — FULL JSON INPUT (Random Project Proposal)

> **ONLY this JSON controls the PPT**

In [58]:
# ==========================================
# 🟦 MASTER JSON INPUT (20 PAGES)
# ==========================================
# This is the "Source of Truth". 
# Edit this JSON to change content, style, or structure.

raw_json_input = {
  "meta": {
    "title": "Enterprise Project Portfolio",
    "author": "PMO Office",
    "date": "Q4 2025"
  },
  "theme": {
    "colors": {
      "background": [255, 255, 255],
      "title": [30, 60, 114],      # Deep Blue
      "text": [42, 42, 42],        # Dark Gray
      "accent": [42, 82, 152],     # Bright Blue
      "table_header": [30, 60, 114],
      "border": [200, 200, 200]
    },
    "fonts": {
      "family": "Arial",
      "size_title": 32,
      "size_body": 16
    }
  },
  "slides": [
    # --- SECTION 1: PROJECT PROPOSAL ---
    { "type": "cover", "title": "Project Phoenix", "subtitle": "Next-Gen CRM Transformation Proposal" },
    { "type": "content", "title": "Executive Summary", "banner": "Modernizing Customer Relations", "text": "Project Phoenix aims to replace the legacy CRM with a cloud-native solution. This will improve sales efficiency by 30% and reduce maintenance costs by 40%." },
    { "type": "cards", "title": "Strategic Objectives", "cards": [
        { "title": "Efficiency", "items": ["Automate data entry", "Reduce lag time"] },
        { "title": "Intelligence", "items": ["AI-driven insights", "Predictive churn"] },
        { "title": "Scalability", "items": ["Cloud infrastructure", "Global availability"] }
      ]
    },
    { "type": "chart", "title": "Projected ROI Analysis", "chart_type": "COLUMN", "chart": { "categories": ["Year 1", "Year 2", "Year 3"], "series_name": "Savings ($M)", "values": [1.5, 3.2, 5.0] } },
    { "type": "gantt", "title": "Implementation Roadmap", "tasks": [
        { "name": "Discovery", "start_week": 1, "duration": 4, "progress": 100 },
        { "name": "Development", "start_week": 5, "duration": 12, "progress": 20 },
        { "name": "UAT", "start_week": 17, "duration": 4, "progress": 0 }
      ]
    },
    { "type": "table", "title": "Resource Requirements", "columns": [ { "key": "role", "label": "Role" }, { "key": "count", "label": "Count" }, { "key": "cost", "label": "Est. Cost" } ], "rows": [
        { "role": "Solution Architect", "count": "1", "cost": "$120k" },
        { "role": "Senior Dev", "count": "3", "cost": "$300k" },
        { "role": "QA Engineer", "count": "2", "cost": "$160k" }
      ]
    },
    
    # --- SECTION 2: STATUS REPORT ---
    { "type": "cover", "title": "Weekly Status Report", "subtitle": "Week 42 - Project Phoenix" },
    { "type": "cards", "title": "Project Health", "cards": [
        { "title": "Scope", "items": ["On Track", "No creep detected"] },
        { "title": "Schedule", "items": ["At Risk", "Backend delay"] },
        { "title": "Budget", "items": ["On Track", "Under utilization"] },
        { "title": "Quality", "items": ["On Track", "0 Critical Bugs"] }
      ]
    },
    { "type": "table", "title": "Key Milestones", "columns": [ { "key": "milestone", "label": "Milestone" }, { "key": "date", "label": "Due Date" }, { "key": "status", "label": "Status" } ], "rows": [
        { "milestone": "Phase 1 Design", "date": "Oct 15", "status": "Completed" },
        { "milestone": "API Integration", "date": "Nov 01", "status": "Delayed" },
        { "milestone": "Beta Launch", "date": "Dec 15", "status": "Pending" }
      ]
    },
    { "type": "chart", "title": "Defect Density", "chart_type": "LINE", "chart": { "categories": ["Wk 38", "Wk 39", "Wk 40", "Wk 41"], "series_name": "Bugs Found", "values": [12, 15, 8, 4] } },
    { "type": "content", "title": "Risks & Mitigations", "banner": "Top Priority Risks", "text": "1. API Latency: Mitigation involves caching layer implementation.\n2. Resource Churn: Hiring 2 contractors as backup." },
    
    # --- SECTION 3: FINANCIALS ---
    { "type": "cover", "title": "Q4 Financial Review", "subtitle": "Budget vs Actuals" },
    { "type": "table", "title": "Budget Variance", "columns": [ { "key": "category", "label": "Category" }, { "key": "budget", "label": "Budget" }, { "key": "actual", "label": "Actual" }, { "key": "var", "label": "Variance" } ], "rows": [
        { "category": "Personnel", "budget": "$500k", "actual": "$480k", "var": "+$20k" },
        { "category": "Software", "budget": "$100k", "actual": "$110k", "var": "-$10k" },
        { "category": "Travel", "budget": "$50k", "actual": "$20k", "var": "+$30k" }
      ]
    },
    { "type": "chart", "title": "Quarterly Spend", "chart_type": "BAR", "chart": { "categories": ["Q1", "Q2", "Q3", "Q4"], "series_name": "Spend ($k)", "values": [150, 180, 210, 240] } },
    { "type": "cards", "title": "Financial Highlights", "cards": [
        { "title": "Total Savings", "items": ["$40k YTD", "Driven by remote work"] },
        { "title": "Forecast", "items": ["Stable", "No major capex planned"] }
      ]
    },
    
    # --- SECTION 4: APPENDIX / EXTRAS ---
    { "type": "content", "title": "Next Steps", "text": "1. Approve Phase 2 Budget\n2. Sign off on UAT Plan\n3. Schedule Steering Committee Review" },
    { "type": "image", "title": "Team Photo", "image_path": "team.jpg", "caption": "The Phoenix Team at Kickoff" }
  ]
}
print("✅ JSON Loaded. Ready to Generate.")

✅ JSON Loaded. Ready to Generate.


## 🟦 CELL 9 — PPT Generator (JSON → PPT)

In [59]:
# ==========================================
# 🟦 MAIN GENERATOR (WITH VALIDATION)
# ==========================================

try:
    # 1. Validate JSON against Pydantic Models
    print("🔍 Validating JSON Structure...")
    config = PresentationConfig(**raw_json_input)
    print("✅ Validation Successful! Generating PPT...")

    # 2. Initialize Presentation
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    
    theme = config.theme

    # 3. Loop through Slides
    for slide_data in config.slides:
        # Create blank slide (Layout 6 is usually blank)
        slide = prs.slides.add_slide(prs.slide_layouts[6])
        
        # Apply Background
        bg = slide.background
        fill = bg.fill
        fill.solid()
        fill.fore_color.rgb = get_rgb(theme.colors.background)

        # Dispatch to Renderers
        if slide_data.type == "cover":
            render_cover(slide, slide_data, theme)
        
        elif slide_data.type == "content":
            add_common_elements(slide, slide_data.title, theme)
            render_content(slide, slide_data, theme)
            
        elif slide_data.type == "table":
            add_common_elements(slide, slide_data.title, theme)
            render_table(slide, slide_data, theme)
            
        elif slide_data.type == "chart":
            add_common_elements(slide, slide_data.title, theme)
            render_chart(slide, slide_data, theme)
            
        elif slide_data.type == "cards":
            add_common_elements(slide, slide_data.title, theme)
            render_cards(slide, slide_data, theme)
            
        elif slide_data.type == "gantt":
            add_common_elements(slide, slide_data.title, theme)
            render_gantt(slide, slide_data, theme)
            
        elif slide_data.type == "image":
            add_common_elements(slide, slide_data.title, theme)
            render_image(slide, slide_data, theme)

    # 4. Save
    output_path = "enterprise_presentation.pptx"
    prs.save(output_path)
    print(f"🎉 Presentation saved to: {output_path}")

    # 5. Download Link
    if os.path.exists(output_path):
        with open(output_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        
        download_link = f'<a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" download="{output_path}" style="font-size: 20px; font-weight: bold; color: blue; background-color: #f0f0f0; padding: 10px; border-radius: 5px; text-decoration: none;">⬇️ DOWNLOAD {output_path}</a>'
        display(HTML(download_link))

except ValidationError as e:
    print("❌ JSON VALIDATION ERROR! Please fix your JSON input.")
    print("---------------------------------------------------")
    # Print a friendly error
    for err in e.errors():
        loc = " -> ".join([str(x) for x in err['loc']])
        print(f"Error in [{loc}]: {err['msg']}")
except Exception as e:
    print(f"❌ UNEXPECTED ERROR: {str(e)}")

🔍 Validating JSON Structure...
✅ Validation Successful! Generating PPT...
🎉 Presentation saved to: enterprise_presentation.pptx


## 🟦 CELL 10 — Save & Download PPT

In [60]:
import base64
import os
from IPython.display import HTML, display

output_path = "project_proposal.pptx"
prs.save(output_path)
print(f"Presentation saved to: {output_path}")

if os.path.exists(output_path):
    with open(output_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    # Create a clickable download link
    download_link = f'<a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" download="{output_path}" style="font-size: 20px; font-weight: bold; color: blue;">⬇️ Click Here to Download PPT</a>'
    display(HTML(download_link))

Presentation saved to: project_proposal.pptx
